# Agents

LLMs need tools to respond to users with information about current events. Many chatbots use a `web-search` tool to get up-to-date information.

## LangChain

LangChain is a framework for building LLM applications. It brings together many existing libraries and integrations under one interface. I will be using LangChain.

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

sys.path.append(str(Path.cwd().parent))
load_dotenv()

llm = ChatOpenAI(model="gpt-4o", temperature=0)

response = llm.invoke("What is the date today?")
print(response.content)


I'm sorry, but I can't provide real-time information. Please check the date on your device or calendar.


## LLMs do not have up-to date information. A datetime tool is needed.

In [2]:
from datetime import date
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a helpful assistant. "
    f"Today's date is {date.today()}. "
    "Answer as accurately as you can."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{user_input}"),
])

chain = prompt | llm

response = chain.invoke({
    "user_input": "Who is the current president of the United States?"
})
response.content

'As of my last update in October 2023, the President of the United States was Joe Biden. However, I do not have information on events or changes that may have occurred after that date. To find out the current president as of September 26, 2026, I recommend checking a reliable and up-to-date news source.'

## Adding a web-search tool

When a LLM has a web-search tool, it can answer with `up-to-date` information.



In [3]:
from typing import Annotated, TypedDict

from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from tools.search import web_search


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


system_prompt = (
    "You are a helpful assistant. "
    "Use the web-search tool for current events. "
    "Call it at most once, then answer using the search results. "
    f"Today's date is {date.today():%A, %B %d, %Y}. "
    "Answer as accurately as you can."
)

llm_with_tools = llm.bind_tools([web_search])


def agent_node(state: AgentState) -> dict[str, list[AnyMessage]]:
    has_used_tool = any(
        isinstance(message, ToolMessage) for message in state["messages"]
    )
    model = llm if has_used_tool else llm_with_tools
    response = model.invoke([
        SystemMessage(content=system_prompt),
        *state["messages"],
    ])
    return {"messages": [response]}


wf = StateGraph(AgentState)
wf.add_node("agent", agent_node)
wf.add_node("tools", ToolNode(tools=[web_search]))
wf.set_entry_point("agent")
wf.add_conditional_edges(
    "agent",
    tools_condition,
    {"tools": "tools", END: END},
)
wf.add_edge("tools", "agent")

app = wf.compile()

result = app.invoke({
    "messages": [
        HumanMessage(content="Who is the current president of the United States?")
    ]
})
result["messages"][-1].content

'The current president of the United States is Donald J. Trump, who began his second, nonconsecutive term on January 20, 2025.'

### About the output

This is essentially what a agent is. A agent can make tool calls to achieve tasks that it was not trained on and can act on its own. That agent, though minor, got that information on the current president of the united states by using DuckDuck go to answer that question.

## Multi-Agent

A Multi-Agent can be one of two forms:

1. Has a central supervisor and controls the sub-agents
2. Has no central supervisor and the sub-agents work, or compete amoung themselves.



In [7]:
from datetime import date
from typing import Literal, NotRequired, TypedDict

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command
from pydantic import BaseModel, Field

from tools.audio import transcribe_audio
from tools.search import web_search

llm = ChatOpenAI(model="gpt-4o", temperature=0)
TODAY = f"{date.today():%B %d, %Y}"
MAX_STEPS = 6  # safety valve so a confused supervisor can't loop forever


class MAState(TypedDict):
    user_request: str
    audio_path: NotRequired[str]
    transcript: NotRequired[str]
    research: NotRequired[str]
    report: NotRequired[str]
    steps: NotRequired[int]


# ---------- workers ----------

def specialist_agent(role: str, tools: list, build_input, output_key: str):
    agent = create_agent(
        llm,
        tools=tools,
        system_prompt=f"{role} Today's date is {TODAY}. You MUST call your tool before answering.",
    )

    def node(state: MAState) -> dict:
        result = agent.invoke({"messages": [HumanMessage(content=build_input(state))]})
        return {output_key: result["messages"][-1].content}

    return node


audio_agent = specialist_agent(
    "You are an audio agent. Transcribe the audio file at the given path "
    "and reply with only the transcript.",
    tools=[transcribe_audio],
    build_input=lambda s: s["audio_path"],
    output_key="transcript",
)

research_agent = specialist_agent(
    "You are an expert research agent. Use the web_search tool to find "
    "up-to-date information relevant to the user's request.",
    tools=[web_search],
    build_input=lambda s: (
        f"Request: {s['user_request']}"
        + (f"\n\nTranscript: {s['transcript']}" if s.get("transcript") else "")
    ),
    output_key="research",
)


def writer_node(state: MAState) -> dict:
    parts = [f"Request: {state['user_request']}"]
    if state.get("transcript"):
        parts.append(f"Transcript: {state['transcript']}")
    if state.get("research"):
        parts.append(f"Research: {state['research']}")
    response = llm.invoke([
        SystemMessage(content=(
            f"Today's date is {TODAY}. Answer the user's request using only the "
            "material provided. If something needed is missing, say so."
        )),
        HumanMessage(content="\n\n".join(parts)),
    ])
    return {"report": response.content}


# ---------- supervisor ----------

WORKERS = ["audio_agent", "research_agent", "writer"]


class Route(BaseModel):
    reasoning: str = Field(description="One sentence on why this is the next step.")
    next: Literal["audio_agent", "research_agent", "writer"]


supervisor_llm = llm.with_structured_output(Route)

SUPERVISOR_PROMPT = f"""You are a supervisor coordinating specialist agents. Today's date is {TODAY}.
Pick the single next agent to run, based on the user's request and what has been gathered so far.

Agents:
- audio_agent: transcribes an audio file. Only useful if an audio file was provided and has not been transcribed yet.
- research_agent: searches the web for current information. Use when the request needs facts, news or anything
  up to date, or when a transcript's topics need looking up. Don't run it again once research exists.
- writer: writes the final answer to the user. Choose it once you have everything the request needs.

Never pick an agent whose output already exists, and never pick audio_agent when no audio file was provided."""


def supervisor(state: MAState) -> Command[Literal["audio_agent", "research_agent", "writer"]]:
    steps = state.get("steps", 0)
    if steps >= MAX_STEPS:
        return Command(goto="writer")

    status = "\n".join([
        f"User request: {state['user_request']}",
        f"Audio file provided: {'yes' if state.get('audio_path') else 'no'}",
        f"Transcript: {'done' if state.get('transcript') else 'not yet'}",
        f"Research: {'done' if state.get('research') else 'not yet'}",
    ])
    decision = supervisor_llm.invoke([
        SystemMessage(content=SUPERVISOR_PROMPT),
        HumanMessage(content=status),
    ])

    # Guardrails: don't trust the LLM to respect preconditions.
    nxt = decision.next
    if nxt == "audio_agent" and (not state.get("audio_path") or state.get("transcript")):
        nxt = "research_agent" if not state.get("research") else "writer"
    if nxt == "research_agent" and state.get("research"):
        nxt = "writer"

    print(f"[supervisor] -> {nxt}: {decision.reasoning}")
    return Command(goto=nxt, update={"steps": steps + 1})


# ---------- graph ----------

ma = StateGraph(MAState)
ma.add_node("supervisor", supervisor)
ma.add_node("audio_agent", audio_agent)
ma.add_node("research_agent", research_agent)
ma.add_node("writer", writer_node)

ma.add_edge(START, "supervisor")
ma.add_edge("audio_agent", "supervisor")      # workers report back
ma.add_edge("research_agent", "supervisor")
ma.add_edge("writer", END)                    # writer ends the run

multi_agent = ma.compile()


if __name__ == "__main__":
    # Audio + news
    result = multi_agent.invoke({
        "user_request": "What is this audio about, and what is the latest news on it?",
        "audio_path": "/home/nick/github-projects/chatbot-website/assets/14 Money (That's What I Want).m4a",
    })
    print(result["report"])

    # Current events only
    result = multi_agent.invoke({"user_request": "What is going on in the news with Nvidia?"})
    print(result["report"])

[supervisor] -> audio_agent: The audio file needs to be transcribed to understand its content before researching the latest news on it.
[supervisor] -> research_agent: The transcript is done, so we need to find the latest news related to its topics.


h2 connection driver error: connection reset


[supervisor] -> writer: All necessary information has been gathered, so it's time to write the final answer.
The audio transcript is of the song "Money (That's What I Want)," which expresses a desire for money over other things, emphasizing that while love is thrilling, it doesn't pay the bills. The song was co-written by Janie Bradford and Berry Gordy and became Motown's first hit single.

The latest news related to this song is the passing of Janie Bradford, who died at the age of 87 on August 21, 2026. She was a pioneering songwriter at Motown and played a significant role in the label's success. Her contributions to the music industry are being celebrated following her passing.
[supervisor] -> research_agent: The user is asking for current news about Nvidia, which requires up-to-date information.
[supervisor] -> writer: The research has been completed, so it's time to write the final answer.
Here are some recent developments regarding Nvidia as of September 2026:

1. **Collaboratio